# Build an MCP Agent

**Prerequisites:** all of `01`–`05`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

Every notebook in this chapter has made the same promise: `02_agent_runtime` and `03_tools` would not have to change. This one is the test, and it is short, because if the promise holds there is almost nothing left to do.

The claim being checked is narrow and worth stating exactly. Not *"an agent can use MCP tools"* — notebook 2 showed that. It is that the loop written four chapters ago, before any of this existed, runs a tool living in somebody else's process **without knowing it does**, and that every guarantee `03_tools` built still means what it meant.

One thing this notebook also does that the others deliberately avoided: it calls a real model. Everything up to now has been measurable without one, which is why the saved outputs have been worth reading. Here the interesting question — *does the refusal a server wrote actually change what the agent does next?* — is one only a live model can answer.

## Setup

In [1]:
import json
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "02_agent_runtime"))
sys.path.append(str(Path.cwd().parent.parent / "03_tools"))

import psutil

from act import Act
from chat import Conversation, count_tokens
from observe import Observe
from registry import Registry
from runtime import Runtime
from think import Think

from remote import remote_tools
from session import Session

REGISTRY_SERVER = Path.cwd().parent / "registry_server.py"

session = Session(REGISTRY_SERVER).__enter__()

registry = Registry()
for t in remote_tools(session):
    registry.add(t)

print("discovered:", registry.names)

discovered: ['weather', 'pressure', 'save_note']


Three tools, none of which exist in this process. `weather` refuses politely when it has no station; `pressure` has a `KeyError` in it; `save_note` is wrapped in `verified()` and truncates what it saves.

## The host decides what the model sees

Notebook 1 named three roles and said the host was the one with no code in the SDK. This is it — and it is two lines.

In [2]:
everything = count_tokens(json.dumps([t.schema for t in registry]))

shown = registry.subset(names=["weather"])
chosen = count_tokens(json.dumps([t.schema for t in shown]))

print(f"connected to: {registry.names}  ({everything} tokens/turn)")
print(f"put in front of the model: {shown.names}  ({chosen} tokens/turn)")

connected to: ['weather', 'pressure', 'save_note']  (236 tokens/turn)
put in front of the model: ['weather']  (81 tokens/turn)


Connecting to a server and putting its tools in front of a model are two decisions, and `subset()` is where they come apart. The protocol will hand you everything a server offers; deciding what belongs in the prompt is a judgement it cannot make for you.

Here it is also keeping the notebook honest. `pressure` is a broken tool and `save_note` writes things — neither belongs in a demonstration about whether a refusal reaches the model, and leaving them in would let the run wander.

## The loop, unchanged

This is `03_tools` notebook 3's runtime, copied. The only difference is where the tools came from.

In [3]:
runtime = Runtime(
    loop=[
        Observe(event="What's the weather in Vestby, Norway?"),
        Think(tools=shown.tools, allow_fork=False),
        Act(functions=shown.invokers(timeout=10, max_tokens=400), schemas=None),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=8,
    max_tokens=8000,
)

conversation = runtime.run(Conversation())

for m in conversation.messages:
    print(f"- {m['role']:9} {str(m.get('content'))[:96]!r}")

- user      "What's the weather in Vestby, Norway?"
- decision  '{"city":"Vestby, Norway"}'
- tool      'No weather station for Vestby, Norway. Try a larger nearby city.'
- decision  'No weather station for Vestby, Norway. Try a larger nearby city.'
- assistant 'No weather station for Vestby, Norway. Try a larger nearby city.'


Read that trace against what each chapter contributed, because every line of it is somebody's guarantee holding:

- the **decision** came from `Think`, which read a schema built by `Tool.schema` in `03_tools`, published over the wire by `expose()` in notebook 4, and turned back into a `Tool` by `remote.py` in notebook 2
- the **tool** message is a `ToolError` raised by a function in another process. `expose()` read `Result.for_the_model`, decided it was for the model, and put it in `content` rather than `_meta`
- `remote.py` read `forge/kind: refused` and raised `ToolError`, so the client's own `invoke()` classified it identically to a local refusal
- and the **assistant** message is the sentence `weather` wrote, reaching a human

That last line is worth sitting with, because it is not the outcome I expected when writing this cell. The tool said *try a larger nearby city*; the plausible next move was for the model to try Oslo. It did not. It passed the advice on and stopped.

That is a defensible answer — the user asked about Vestby, and "there is no station there, try a larger city nearby" is true and actionable by the person who asked — but it is worth being precise that **the model chose to relay rather than to retry**, because the failure mode this chapter has been guarding against looks identical from one turn away. An agent that stalls holding advice it was never shown, and an agent that received the advice and decided the user should see it, produce the same-shaped transcript. Only the middle of the trace tells them apart, which is the argument for `Result.kind` being a thing you can inspect rather than a thing you infer from behaviour.

What the cell does settle — and what no amount of offline testing could have — is that the sentence survived. A string written by a function in another process, for a caller it will never meet, crossed a protocol, two `invoke()` calls and a summariser without being flattened into "the tool failed". Every notebook in this chapter was in service of that, and it is the one thing only a live model could have confirmed.

## Nothing changed in chapters 2 and 3

Worth checking rather than asserting.

In [4]:
import ast
import inspect

import act
import invoke as invoke_module
import registry as registry_module
import runtime as runtime_module
import think
import tools as tools_module

def imports(module):
    """Top-level package names this module imports. Not a text search -- a mention of `mcp`
    in a comment is not a dependency, and this cell is about dependencies."""
    found = set()
    for node in ast.walk(ast.parse(inspect.getsource(module))):
        if isinstance(node, ast.Import):
            found.update(a.name.split(".")[0] for a in node.names)
        elif isinstance(node, ast.ImportFrom) and node.module and node.level == 0:
            found.add(node.module.split(".")[0])
    return found

for mod in (act, runtime_module, think, tools_module, registry_module, invoke_module):
    deps = imports(mod)
    print(f"{mod.__name__ + '.py':14} imports mcp? {str('mcp' in deps):<6} ({len(deps)} imports)")

print()
print("what Act was actually handed:", type(shown.invokers()["weather"]).__name__,
      "-- an ordinary callable")

act.py         imports mcp? False  (6 imports)
runtime.py     imports mcp? False  (7 imports)
think.py       imports mcp? False  (6 imports)
tools.py       imports mcp? False  (3 imports)
registry.py    imports mcp? False  (4 imports)
invoke.py      imports mcp? False  (6 imports)

what Act was actually handed: function -- an ordinary callable


`Act` calls a function. The function happens to send JSON-RPC down a pipe to a second Python process, and nothing in `02_agent_runtime` has any way to find that out, or any reason to want to.

## The gap: where does the session live?

There is one thing this chapter genuinely needs and does not have, and it is best seen by asking where the live session is being kept.

In [5]:
weather_tool = registry["weather"]

print("free variables of the tool's function:", weather_tool.function.__code__.co_freevars)
print()
print("anything on the Conversation holding it?",
      [a for a in vars(conversation) if "session" in a.lower() or "mcp" in a.lower()] or "nothing")

free variables of the tool's function: ('mcp_tool', 'session')

anything on the Conversation holding it? nothing


The session is closed over by the function. That is the only place it exists.

It works, and it is fragile in a specific way. `Conversation` is what the runtime passes from component to component — it carries messages, `pending`, budgets, branches. It has no slot for a **connection**, so a live MCP session has nowhere to live except inside the closure of every tool that uses it. Which means:

- the session's lifetime is not the conversation's lifetime; it is whatever scope built the tools
- a component that wanted to reconnect a dead server could not reach the session to do it
- handing a conversation to another thread hands over tools pointing at a session that thread does not own

`04_memory` notebook 5 found exactly this and described it in almost the same words — *nothing on `Conversation` holds a memory handle, so `remember` has to close over it.* Two chapters arriving independently at the same missing thing is the strongest argument either of them makes: **`Conversation` needs somewhere to keep resources, not just data.** Neither chapter adds it, because a field invented twice from two directions should be designed once, deliberately, with both uses on the table.

## When the server dies

The last thing an agent has to survive. Kill the server mid-run and ask again.

In [6]:
child = next(c for c in psutil.Process().children(recursive=True)
             if "python" in c.name().lower())
print("killing the server process:", child.pid)
child.kill()
time.sleep(1.0)

after = Conversation()
after.messages.append({"role": "decision", "type": "tool", "tool": "weather",
                       "content": json.dumps({"city": "Oslo"})})
Act(functions=shown.invokers(timeout=10), schemas=None).run(after)

print("into the conversation:", after.pending["content"])

killing the server process: 24212


into the conversation: weather could not be completed.


`weather could not be completed.` — and nothing else.

Trace why that is the right answer. The transport raised, because the pipe is gone. `remote.py` did not catch it, so it reached `invoke()`'s `except Exception` arm and became `broken`. `broken` is the one kind whose `for_the_model` is False, so `invoker()` replaced the message with an acknowledgement and moved the cause to `Result.value`.

Which is what you want. A dead server is not something a model can fix, and telling it `MCPError: Connection closed` invites it to invent a workaround for a fault it cannot see. Notebook 2 chose `refused` as the default reading of an *unexplained* failure precisely so that this case — an explained one, explained by the transport itself — could stay `broken`.

What the agent does next is a policy question this chapter does not answer: retry the server, fall back to another, or stop. All three need the thing the previous section said is missing — a place to keep the connection where something other than a closure can reach it.

In [7]:
session.__exit__(None, None, None)
print("session closed")

session closed


## Where this leaves the chapter

The promise held. Across six notebooks, the changes to earlier chapters were:

| | |
|---|---|
| `02_agent_runtime` | **none** |
| `03_tools` | one: `invoker()` now honours `for_the_model`, a flag it already computed and nobody read |

That second one was not an MCP requirement. It was a latent bug that only started to matter once `broken` stopped meaning "a function in this repo has a bug" and started meaning "a server we do not control declined to explain itself".

And the sentence notebook 1 spent its length earning still holds at the end: **MCP does not change what a tool is, it changes who owns the process it runs in.** Every difficulty in this chapter — one bool instead of six kinds, a schema more permissive than we would have written, verification that only works on the far side, an environment handed over at launch — is a consequence of ownership, not of protocol design.

What is deliberately unfinished:

- **`Conversation` has no slot for a connection.** Found here and in `04_memory`, left for whichever chapter designs it properly
- **Reconnection and failover.** Needs the above first
- **Authentication.** Irrelevant for stdio, unavoidable for HTTP, and `08_guardrails`' problem
- **The environment a server starts with.** `05_mcp_transport` measured it and added `inherit=False`; deciding what a tool should be *allowed* to reach is a guardrails question, and the argument is already there waiting